CAELI - Kunstmatige vierkanten (gaten) maken in AOD-data van één orbit
Doel: op plekken waar al waarden zijn, vierkanten met ontbrekende waarden maken
(met losse resterende pixels erin), zodat je daar a model op kunt trainen.

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.ndimage import gaussian_filter
from matplotlib.patches import Rectangle

# 1. Instellingen

In [ ]:
orbit = "2026-07-29 08:25:00.000000"
csv_pad = "29Juli_Unfiltered.csv"
AOD_COL = "optical_depth_055"

# Pixelgrootte in graden (~1 km). Pas aan als je echte resolutie anders is.
STEP = 0.01

# Parameters voor de kunstmatige vierkanten
N_SQUARES = 20
SIZE_RANGE = (30, 60)         # zijde van het vierkant in pixels
ASPECT_JITTER = 5             # max verschil breedte/hoogte in pixels
MIN_VALID_FRAC = 0.7          # blok moet minstens zo vol zijn met echte waarden
KEEP_FRAC_RANGE = (0.02, 0.30)  # aandeel pixels dat binnen het blok blijft staan
HOLE_SMOOTHNESS = (1, 3)        # sigma: hoger = grotere restvlekjes in het blok
SEED = 42

MAX_GRID_CELLS = 50_000_000     # veiligheidsgrens tegen een te groot grid

# 2. Functie: kunstmatige vierkanten

In [ ]:
def make_artificial_squares(
    grid,
    n_squares=10,
    size_range=(25, 50),
    aspect_jitter=5,
    min_valid_frac=0.9,
    keep_frac_range=(0.02, 0.3),
    hole_smoothness=(1, 3),
    seed=None,
):
    rng = np.random.default_rng(seed)
    out = grid.copy()
    valid = ~np.isnan(grid)
    art_mask = np.zeros(grid.shape, dtype=bool)
    boxes = []   # (rij, kolom, hoogte, breedte) van elk geplaatst vierkant
    H, W = grid.shape

    placed, tries = 0, 0
    while placed < n_squares and tries < n_squares * 500:
        tries += 1

        h = int(rng.integers(size_range[0], size_range[1] + 1))
        w = int(np.clip(h + rng.integers(-aspect_jitter, aspect_jitter + 1),
                        size_range[0], size_range[1]))
        if h >= H or w >= W:
            continue

        r = int(rng.integers(0, H - h))
        c = int(rng.integers(0, W - w))

        # blok moet grotendeels uit echte waarden bestaan
        if valid[r:r + h, c:c + w].mean() < min_valid_frac:
            continue

        # niet over eerdere kunstmatige blokken heen
        if art_mask[max(r - 2, 0):r + h + 2, max(c - 2, 0):c + w + 2].any():
            continue

        # ruimtelijk gecorreleerd patroon van pixels die blijven staan
        noise = gaussian_filter(
            rng.random((h, w)),
            sigma=rng.uniform(*hole_smoothness)
        )
        keep_frac = rng.uniform(*keep_frac_range)
        keep = noise >= np.quantile(noise, 1 - keep_frac)

        removed = (~keep) & valid[r:r + h, c:c + w]
        out[r:r + h, c:c + w][removed] = np.nan
        art_mask[r:r + h, c:c + w] |= removed
        boxes.append((r, c, h, w))
        placed += 1

    print(f"{placed} van {n_squares} vierkanten geplaatst ({tries} pogingen).")

    targets = grid[art_mask]
    return out, art_mask, targets, boxes

# 3. Data inladen en filteren op orbit

In [ ]:
print("CSV inladen...")
dataset = pd.read_csv(csv_pad)
dataset["idx_utc"] = pd.to_datetime(dataset["idx_utc"])

if AOD_COL not in dataset.columns:
    raise SystemExit(
        f"Kolom '{AOD_COL}' niet gevonden. Beschikbaar: {dataset.columns.tolist()}"
    )

orbit_data = dataset[
    dataset["idx_utc"] == pd.to_datetime(orbit)
].copy()

orbit_data = orbit_data.dropna(subset=[AOD_COL, "idx_lon", "idx_lat"])

if orbit_data.empty:
    raise SystemExit(f"Geen metingen gevonden voor orbit: {orbit}")

print(f"{len(orbit_data):,} metingen in deze orbit")

# 4. Losse punten omzetten naar een 2D-grid (nodig voor de vierkanten-functie)

In [ ]:
lon_min, lat_min = orbit_data["idx_lon"].min(), orbit_data["idx_lat"].min()

orbit_data["col"] = ((orbit_data["idx_lon"] - lon_min) / STEP).round().astype(int)
orbit_data["row"] = ((orbit_data["idx_lat"] - lat_min) / STEP).round().astype(int)

n_rows = int(orbit_data["row"].max()) + 1
n_cols = int(orbit_data["col"].max()) + 1
print(f"Grid wordt {n_rows} x {n_cols} = {n_rows * n_cols:,} cellen")

if n_rows * n_cols > MAX_GRID_CELLS:
    raise SystemExit("Grid is te groot, check STEP en je lon/lat-waarden.")

grid = np.full((n_rows, n_cols), np.nan)
grid[orbit_data["row"].values, orbit_data["col"].values] = orbit_data[AOD_COL].values

# 5. Kunstmatige vierkanten maken

In [ ]:
print("Kunstmatige vierkanten maken...")
grid_art, art_mask, targets, boxes = make_artificial_squares(
    grid,
    n_squares=N_SQUARES,
    size_range=SIZE_RANGE,
    aspect_jitter=ASPECT_JITTER,
    min_valid_frac=MIN_VALID_FRAC,
    keep_frac_range=KEEP_FRAC_RANGE,
    hole_smoothness=HOLE_SMOOTHNESS,
    seed=SEED,
)

# Koppel het kunstmatige maskerscherm terug aan de losse datapunten in orbit_data
# Zo kunnen we plt.scatter gebruiken ipv imshow, wat veel scherper oogt!
orbit_data["is_art_removed"] = [
    art_mask[r, c] for r, c in zip(orbit_data["row"], orbit_data["col"])
]
orbit_data_art = orbit_data[~orbit_data["is_art_removed"]].copy()

# 6. Plotten met 2x2 layout en nette ruimte voor de titels

In [ ]:
# Kleurenschaal bepalen op basis van percentielen (1% en 99%)
vmin = orbit_data[AOD_COL].quantile(0.01)
vmax = orbit_data[AOD_COL].quantile(0.99)

fig, axes = plt.subplots(2, 2, figsize=(14, 12), dpi=300, sharex=False, sharey=False)

ax1 = axes[0, 0] # Linksboven: Origineel
ax2 = axes[0, 1] # Rechtsboven: Met kunstmatige vierkanten
ax3 = axes[1, 0] # Linksonder: Masker
ax4 = axes[1, 1] # Rechtsonder: Gereserveerd voor de legenda

# Plot 1: Origineel
kaart1 = ax1.scatter(
    orbit_data["idx_lon"], orbit_data["idx_lat"],
    c=orbit_data[AOD_COL], cmap="viridis", vmin=vmin, vmax=vmax,
    marker="s", s=1, linewidths=0
)
ax1.set_title("Origineel", fontsize=12, fontweight="bold", pad=10)

# Plot 2: Met kunstmatige vierkanten
kaart2 = ax2.scatter(
    orbit_data_art["idx_lon"], orbit_data_art["idx_lat"],
    c=orbit_data_art[AOD_COL], cmap="viridis", vmin=vmin, vmax=vmax,
    marker="s", s=1, linewidths=0
)
ax2.set_title("Met kunstmatige vierkanten", fontsize=12, fontweight="bold", pad=10)

# Rode kaders tekenen om de geplaatste vierkanten
for (r, c, h, w) in boxes:
    rect_lon = lon_min + c * STEP - STEP / 2
    rect_lat = lat_min + r * STEP - STEP / 2
    ax2.add_patch(Rectangle(
        (rect_lon, rect_lat), w * STEP, h * STEP,
        fill=False, edgecolor="red", linewidth=0.8
    ))

# Plot 3: Masker (alleen de weggehaalde punten tonen in rood)
removed_data = orbit_data[orbit_data["is_art_removed"]]
ax3.scatter(
    removed_data["idx_lon"], removed_data["idx_lat"],
    color="red", marker="s", s=1, linewidths=0
)
ax3.set_title("Masker (weggehaald)", fontsize=12, fontweight="bold", pad=10)
ax3.set_facecolor("#f0f0f0")

# Assen en labels instellen voor de actieve plots
for ax in [ax1, ax2, ax3]:
    ax.set_xlabel("Longitude", fontsize=10)
    ax.set_ylabel("Latitude", fontsize=10)
    ax.grid(alpha=0.2, linestyle="--")
    ax.set_xlim(orbit_data["idx_lon"].min(), orbit_data["idx_lon"].max())
    ax.set_ylim(orbit_data["idx_lat"].min(), orbit_data["idx_lat"].max())

# Plot 4 (rechtsonder) verbergen en gebruiken voor de kleurenbalk
ax4.axis("off")

cbar = fig.colorbar(kaart2, ax=ax4, orientation="horizontal", fraction=0.4, pad=0.1)
cbar.set_label("Optical Depth 0.55 µm", fontsize=11)

fig.suptitle(f"Aerosol Optical Depth - Orbit {orbit}", fontsize=15, y=0.97, fontweight="bold")

# rect=[left, bottom, right, top] laat ruimte vrij aan de bovenkant voor de hoofdtitel
plt.tight_layout(rect=[0, 0, 1, 0.94])
plt.show()


# 7. Samenvatting

In [ ]:
print(f"Orbit: {orbit}")
print(f"Grid: {grid.shape[0]} x {grid.shape[1]} pixels")
print(f"Geldige pixels origineel: {len(orbit_data):,}")
print(f"Kunstmatig weggehaald:    {len(removed_data):,}")
print(f"Targets (waarheid):       {len(targets):,}")